# Stage 5 — Governance: the whole platform, defended

**Checkpoint:** `stage-5`

Five modules, one score spine, and a documentation pack. This notebook is the view a **model
validator** would want: every gate in one table, and one borrower traced through every decision
the platform makes about them.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "docs/model_doc_pack/MODEL_DOCUMENTATION.md",
    "score/models/metadata.json",
    "adjudication/models/metadata.json",
    "ews/models/metadata.json",
    "line_increase/models/metadata.json",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-5 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-5      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-5 artifacts present.")

## 1. Every gate in the platform, on one page

In [ ]:
import json

def meta(p):
    try:    return json.loads(Path(p).read_text())
    except Exception: return {}

def row(module, path, metric_key, gate_key, note="gated"):
    d = meta(path)
    return {"module": module, "metric": metric_key,
            "gate": d.get("gate", {}).get(gate_key),
            "actual": d.get("metrics", {}).get(metric_key),
            "status": note}

gates = pd.DataFrame([
    row("Score spine",   "score/models/metadata.json",         "auc",                "auc_min"),
    row("Adjudication",  "adjudication/models/metadata.json",  "auc",                "auc_min"),
    row("Adjudication",  "adjudication/models/metadata.json",  "top20_lift",         "lift_min"),
    row("Early warning", "ews/models/metadata.json",           "top_decile_capture", "capture_min"),
    row("Early warning", "ews/models/metadata.json",           "auc",                "__none__", "REPORTED, not gated"),
    row("Line increase", "line_increase/models/metadata.json", "auc",                "auc_min"),
    row("Line increase", "line_increase/models/metadata.json", "top20_lift",         "lift_min"),
])

def verdict(r):
    if r["gate"] is None or not isinstance(r["gate"], (int, float)):
        return "—"
    return "PASS" if float(r["actual"]) >= float(r["gate"]) else "FAIL"

gates["pass"] = gates.apply(verdict, axis=1)
gates.style.format({"gate": "{:.4f}", "actual": "{:.4f}"})

> **The one that teaches the most is the row marked *reported, not gated*.** Someone measured
> the achievable ceiling, found the target unreachable given the noise in the data, and wrote
> that down instead of tuning until something passed. That sentence is what a validator is
> actually looking for.

In [ ]:
g = gates[gates["gate"].apply(lambda v: isinstance(v, (int, float)))].copy()
g["label"] = g["module"] + "\n" + g["metric"]
fig, ax = plt.subplots(figsize=(10, 4.4))
x = np.arange(len(g))
ax.bar(x - .2, g["gate"].astype(float), width=.4, label="gate", color="#c9c9c9")
ax.bar(x + .2, g["actual"].astype(float), width=.4, label="actual",
       color=["#2f7d4f" if p == "PASS" else "#b3372e" for p in g["pass"]])
ax.set_xticks(x, g["label"], fontsize=8)
ax.legend(); ax.set_title("Gates vs actuals across the platform")
plt.tight_layout(); plt.show()

## 2. One borrower, every decision the platform makes about them

In [ ]:
import joblib
from shared.config import RAW
from score.src.predict import predict_score_pd
from adjudication.src.feature_engineering import ADJ_FEATURE_COLUMNS, compute_adjudication_features
from adjudication.src.policy import PolicyConfig, decide
from pricing.src.portfolio import price_population
from ews.src import watchlist as ews_watchlist
from line_increase.src import candidates as li_candidates

businesses = pd.read_parquet(RAW / "businesses.parquet")
scores = predict_score_pd(businesses)
model  = joblib.load("adjudication/models/adjudication_model.pkl")
config = PolicyConfig.from_dict(json.loads(Path("adjudication/models/policy_config.json").read_text()))
X_all  = compute_adjudication_features(businesses)
dec    = decide(X_all, model.predict_proba(X_all[ADJ_FEATURE_COLUMNS])[:, 1], config)

priced = price_population().set_index("business_id")
ews    = ews_watchlist.score_population()
li     = li_candidates.score_population()

# pick someone who appears everywhere: on book, watchlisted, and offered
on_book = set(priced.index.astype(str))
watched = set(ews["business_id"].astype(str))
bid = next(iter(on_book & watched))

i = businesses.index[businesses["business_id"].astype(str) == bid][0]
print(f"=== Entity 360: {bid} ===\n")
print(f"industry           : {businesses.loc[i, 'industry']}")
print(f"score / band       : {scores.loc[i, 'business_score']}  ({scores.loc[i, 'score_band']})")
print(f"modelled PD        : {scores.loc[i, 'pd']:.2%}")
print(f"decision           : {dec.loc[i, 'decision']}  {dec.loc[i, 'decision_reasons']}")
if bid in priced.index:
    r = priced.loc[bid]
    print(f"quoted / recommended: {r['quoted_rate']:.2%} / {r['recommended_rate']:.2%}"
          f"   {'MISPRICED' if r['mispriced'] else 'priced ok'}")
w = ews.loc[ews['business_id'].astype(str) == bid]
if len(w):
    print(f"watchlist          : {w.iloc[0]['risk_tier']} tier, p={w.iloc[0]['prob']:.1%}, "
          f"triggers={', '.join(w.iloc[0]['triggers']) or 'none'}")
l = li.loc[li.index.astype(str) == bid] if li.index.name else pd.DataFrame()
print("\nOne entity. One score. Five decisions.")

## 3. The documentation pack

In [ ]:
doc = Path("docs/model_doc_pack/MODEL_DOCUMENTATION.md").read_text()
print(f"{len(doc):,} characters, {doc.count(chr(10)):,} lines\n")
print("Sections:")
for line in doc.splitlines():
    if line.startswith("#"):
        print("  " + line)

## 4. The questions to be ready for

A validator is *adversarially curious*, not hostile. Rehearse these:

1. Why is your top variable in the model, and what would make you remove it?
2. Which of the five modules would you trust least, and what would you check first?
3. Show me a decision you disagree with, and tell me why the system made it.
4. What is the ceiling on the early-warning model, and how do you know?
5. If the data drifted, which gate would fail first?

---

**You are at the end.** Everything in this repo you can now rebuild, explain, and defend.